# Linear MPLE with similarity-weighted neighbor averages for DGL Fraud Yelp

We fit the linear pseudo-likelihood model

$$
\mathbb P_{\beta,\gamma}(\sigma_i=1\mid \sigma_{-i},X,G)
=
\operatorname{logit}^{-1}(2h_i),
$$

where

$$
h_i
=
\sum_{r=1}^3 \beta_r \overline S^{\,w}_{ir}
+
2X_i^\top\gamma.
$$

For relation \(r\),

$$
\overline S^{\,w}_{ir}
=
\frac{
\sum_{j:(j,i)\in E_r} w^{(r)}_{ij}\sigma_j^{\mathrm{obs}}
}{
\sum_{j:(j,i)\in E_r} w^{(r)}_{ij}\mathbf 1\{j\text{ observed}\}+\varepsilon
}.
$$

The default edge weight is feature-RBF similarity:

$$
w^{(r)}_{ij}
=
\exp\left(
-\frac{\|X_i-X_j\|^2}{2\tau_r^2}
\right).
$$

The three fitted coefficients are

$$
(\beta_1,\beta_2,\beta_3)
=
(\beta_{\texttt{net\_rur}},\beta_{\texttt{net\_rsr}},\beta_{\texttt{net\_rtr}}).
$$

The notebook supports two beta modes:

$$
\beta_r\in\mathbb R
\quad\text{or}\quad
\beta_r\ge 0.
$$

Set `BETA_CONSTRAINT = "unconstrained"` or `BETA_CONSTRAINT = "nonnegative"` below.


In [ ]:
# If needed, install dependencies first. Pick the DGL wheel matching your PyTorch version.
# %pip install -q scikit-learn pandas matplotlib
# %pip install -q dgl -f https://data.dgl.ai/wheels/torch-2.1/repo.html


In [1]:
import math
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

import matplotlib.pyplot as plt

import dgl
from dgl.data import FraudYelpDataset


SEED = 717

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

## Load Fraud Yelp

The original labels are \(y_i\in\{0,1\}\). We convert them to spins by

$$
\sigma_i=2y_i-1.
$$


In [2]:
dataset = FraudYelpDataset(
    random_seed=SEED,
    train_size=0.7,
    val_size=0.1,
    verbose=True,
)

g = dataset[0]

print(g)
print("node types:", g.ntypes)
print("edge types:", g.etypes)
print("canonical edge types:", g.canonical_etypes)


Done loading data from cached files.
Graph(num_nodes={'review': 45954},
      num_edges={('review', 'net_rsr', 'review'): 6805486, ('review', 'net_rtr', 'review'): 1147232, ('review', 'net_rur', 'review'): 98630},
      metagraph=[('review', 'review', 'net_rsr'), ('review', 'review', 'net_rtr'), ('review', 'review', 'net_rur')])
node types: ['review']
edge types: ['net_rsr', 'net_rtr', 'net_rur']
canonical edge types: [('review', 'net_rsr', 'review'), ('review', 'net_rtr', 'review'), ('review', 'net_rur', 'review')]


In [3]:
def get_node_data(g, key, ntype=None):
    if ntype is None:
        if len(g.ntypes) != 1:
            raise ValueError("Pass ntype explicitly when the graph has multiple node types.")
        ntype = g.ntypes[0]

    value = g.ndata[key]
    if isinstance(value, dict):
        return value[ntype]
    return value


ntype = g.ntypes[0]

X_raw = get_node_data(g, "feature", ntype).float()
y01 = get_node_data(g, "label", ntype).long()

train_mask = get_node_data(g, "train_mask", ntype).bool()
val_mask = get_node_data(g, "val_mask", ntype).bool()
test_mask = get_node_data(g, "test_mask", ntype).bool()

sigma_all = 2.0 * y01.float() - 1.0

summary = pd.DataFrame({
    "split": ["train", "validation", "test", "all"],
    "n": [
        int(train_mask.sum()),
        int(val_mask.sum()),
        int(test_mask.sum()),
        int(y01.numel()),
    ],
    "positives_label_1": [
        int(y01[train_mask].sum()),
        int(y01[val_mask].sum()),
        int(y01[test_mask].sum()),
        int(y01.sum()),
    ],
})
summary["positive_rate"] = summary["positives_label_1"] / summary["n"]
summary


,split,n,positives_label_1,positive_rate
0,train,32167,4726,0.146921
1,validation,4595,651,0.141676
2,test,9192,1300,0.141427
3,all,45954,6677,0.145297


In [4]:
EDGE_TYPES = ["net_rur", "net_rsr", "net_rtr"]

BETA_LABELS = [
    "beta_1: net_rur",
    "beta_2: net_rsr",
    "beta_3: net_rtr",
]

missing = [e for e in EDGE_TYPES if e not in g.etypes]
if missing:
    raise ValueError(f"Missing expected edge types: {missing}. Present edge types: {g.etypes}")

for etype in EDGE_TYPES:
    print(f"{etype:8s}: {g.num_edges(etype=etype):,} directed edges")


net_rur : 98,630 directed edges
net_rsr : 6,805,486 directed edges
net_rtr : 1,147,232 directed edges


## Standardize features

We standardize the 32 node features using the training nodes only. Then we add an intercept column for \(X_i^\top\gamma\).


In [5]:
def standardize_with_train_stats(X, train_mask, add_intercept=True, eps=1e-12):
    X = X.float()
    mean = X[train_mask].mean(dim=0)
    std = X[train_mask].std(dim=0, unbiased=False).clamp_min(eps)
    X_std = (X - mean) / std

    if add_intercept:
        ones = torch.ones((X_std.shape[0], 1), dtype=X_std.dtype)
        X_aug = torch.cat([ones, X_std], dim=1)
    else:
        X_aug = X_std

    return X_aug, X_std, mean, std


X_aug, X_stdzd, X_mean, X_std = standardize_with_train_stats(
    X_raw,
    train_mask,
    add_intercept=True,
)

print("raw feature shape:", tuple(X_raw.shape))
print("standardized feature shape:", tuple(X_stdzd.shape))
print("augmented feature shape:", tuple(X_aug.shape))


raw feature shape: (45954, 32)
standardized feature shape: (45954, 32)
augmented feature shape: (45954, 33)


## Compute similarity-weighted neighbor averages

By default, only training labels are treated as observed:

$$
\sigma_j^{\mathrm{obs}}
=
\begin{cases}
\sigma_j, & j\in\mathcal T,\\
0, & j\notin\mathcal T.
\end{cases}
$$

This avoids using validation/test labels as covariates.

The denominator may either average over observed labeled source nodes only, or over all graph neighbors. The cleaner semi-supervised choice is observed-only denominator.


In [24]:
SOURCE_LABEL_MASK_FOR_NEIGHBOR_AVERAGES = train_mask
# SOURCE_LABEL_MASK_FOR_NEIGHBOR_AVERAGES = torch.ones_like(train_mask, dtype=torch.bool)  # oracle diagnostic only

SIMILARITY_MODE = "cosine_positive"  # one of {"rbf_features", "cosine_positive"}

# Chunk size controls memory use. Reduce this if you run out of memory.
EDGE_CHUNK_SIZE = 500_000

# Number of edges sampled per relation to estimate the RBF scale tau_r.
TAU_SAMPLE_SIZE = 200_000

# If True, denominator averages only over observed-label source nodes.
DENOMINATOR_USES_ONLY_OBSERVED_LABELS = True


@torch.no_grad()
def estimate_relation_taus_from_edge_samples(
    g,
    X_for_similarity,
    edge_types,
    sample_size=200_000,
    seed=717,
    device=None,
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    rng = np.random.default_rng(seed)
    X_dev = X_for_similarity.to(device).float()

    tau_values = []

    for etype in edge_types:
        src, dst = g.edges(etype=etype)
        m = src.numel()

        if m == 0:
            tau_values.append(1.0)
            continue

        k = min(sample_size, m)
        sample_idx = rng.choice(m, size=k, replace=False)

        src_s = src[sample_idx].to(device)
        dst_s = dst[sample_idx].to(device)

        diff = X_dev[src_s] - X_dev[dst_s]
        dist = diff.pow(2).sum(dim=1).sqrt()

        tau = torch.median(dist).item()
        if not np.isfinite(tau) or tau <= 1e-12:
            tau = float(dist.mean().item())
        if not np.isfinite(tau) or tau <= 1e-12:
            tau = 1.0

        tau_values.append(float(tau))

    return torch.tensor(tau_values, dtype=torch.float32)


@torch.no_grad()
def edge_similarity_weights(X_dev, src, dst, mode, tau=None):
    if mode == "rbf_features":
        if tau is None:
            raise ValueError("tau is required for rbf_features similarity.")
        diff = X_dev[src] - X_dev[dst]
        dist2 = diff.pow(2).sum(dim=1)
        w = torch.exp(-dist2 / (2.0 * float(tau) ** 2))
        return w

    if mode == "cosine_positive":
        x_src = X_dev[src]
        x_dst = X_dev[dst]
        numerator = (x_src * x_dst).sum(dim=1)
        denom = x_src.norm(dim=1).clamp_min(1e-12) * x_dst.norm(dim=1).clamp_min(1e-12)
        cosine = numerator / denom
        return torch.clamp(cosine, min=0.0, max=1.0)

    raise ValueError("SIMILARITY_MODE must be one of {'rbf_features', 'cosine_positive'}.")


@torch.no_grad()
def compute_similarity_weighted_neighbor_averages(
    g,
    sigma_all,
    observed_mask,
    X_for_similarity,
    edge_types,
    similarity_mode="rbf_features",
    tau_values=None,
    denominator_observed_only=True,
    chunk_size=500_000,
    device=None,
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    n = sigma_all.numel()

    X_dev = X_for_similarity.to(device).float()
    sigma_dev = sigma_all.to(device).float()
    obs_dev = observed_mask.to(device).bool()

    avg_columns = []
    numerator_columns = []
    denominator_columns = []

    for r, etype in enumerate(edge_types):
        src_cpu, dst_cpu = g.edges(etype=etype)
        m = src_cpu.numel()

        numerator = torch.zeros(n, dtype=torch.float32, device=device)
        denominator = torch.zeros(n, dtype=torch.float32, device=device)

        tau_r = None
        if similarity_mode == "rbf_features":
            tau_r = float(tau_values[r])

        for start in range(0, m, chunk_size):
            end = min(start + chunk_size, m)

            src = src_cpu[start:end].to(device)
            dst = dst_cpu[start:end].to(device)

            w = edge_similarity_weights(
                X_dev,
                src,
                dst,
                mode=similarity_mode,
                tau=tau_r,
            )

            obs_src = obs_dev[src].float()

            weighted_spin = w * obs_src * sigma_dev[src]
            numerator.index_add_(0, dst, weighted_spin)

            if denominator_observed_only:
                denominator_increment = w * obs_src
            else:
                denominator_increment = w

            denominator.index_add_(0, dst, denominator_increment)

        avg = numerator / denominator.clamp_min(1e-12)
        avg = torch.where(denominator > 0, avg, torch.zeros_like(avg))

        avg_columns.append(avg.cpu())
        numerator_columns.append(numerator.cpu())
        denominator_columns.append(denominator.cpu())

    Sbar_w = torch.stack(avg_columns, dim=1)
    N_w = torch.stack(numerator_columns, dim=1)
    D_w = torch.stack(denominator_columns, dim=1)

    return Sbar_w, N_w, D_w


## Estimate RBF bandwidths \(\tau_r\)


In [25]:
tau_values = estimate_relation_taus_from_edge_samples(
    g=g,
    X_for_similarity=X_stdzd,
    edge_types=EDGE_TYPES,
    sample_size=TAU_SAMPLE_SIZE,
    seed=SEED,
    device=device,
)

pd.DataFrame({
    "edge_type": EDGE_TYPES,
    "tau_r": tau_values.numpy(),
})


,edge_type,tau_r
0,net_rur,6.746763
1,net_rsr,7.590441
2,net_rtr,7.569873


## Build \(\overline S^{\,w}\)


In [26]:
Sbar_w, numerator_w, denominator_w = compute_similarity_weighted_neighbor_averages(
    g=g,
    sigma_all=sigma_all,
    observed_mask=SOURCE_LABEL_MASK_FOR_NEIGHBOR_AVERAGES,
    X_for_similarity=X_stdzd,
    edge_types=EDGE_TYPES,
    similarity_mode=SIMILARITY_MODE,
    tau_values=tau_values,
    denominator_observed_only=DENOMINATOR_USES_ONLY_OBSERVED_LABELS,
    chunk_size=EDGE_CHUNK_SIZE,
    device=device,
)

pd.DataFrame(Sbar_w[train_mask].numpy(), columns=EDGE_TYPES).describe().T


,count,mean,std,min,25%,50%,75%,max
net_rur,32167.0,-0.373472,0.519389,-1.0,-1.000000,0.000000,0.000000,1.0
net_rsr,32167.0,-0.714389,0.294284,-1.0,-0.901667,-0.797873,-0.633094,1.0
net_rtr,32167.0,-0.690636,0.399517,-1.0,-1.000000,-0.843196,-0.528417,1.0


In [27]:
neighbor_summary = pd.DataFrame({
    "edge_type": EDGE_TYPES,
    "mean_weighted_denominator_train": denominator_w[train_mask].mean(dim=0).numpy(),
    "median_weighted_denominator_train": denominator_w[train_mask].median(dim=0).values.numpy(),
    "fraction_train_with_no_observed_weighted_neighbors": (denominator_w[train_mask] <= 0).float().mean(dim=0).numpy(),
})

neighbor_summary


,edge_type,mean_weighted_denominator_train,median_weighted_denominator_train,fraction_train_with_no_observed_weighted_neighbors
0,net_rur,0.418040,0.000000,0.590108
1,net_rsr,13.915053,12.457725,0.002394
2,net_rtr,2.445546,1.973895,0.025150


Because \(\overline S^{\,w}_{ir}\in[-1,1]\), we do not standardize it further. The fitted \(\beta_r\)'s are directly coefficients of the weighted average neighbor spins.


## Linear MPLE class with optional beta constraint

Set

```python
BETA_CONSTRAINT = "unconstrained"
```

to allow each \(\beta_r\in\mathbb R\).

Set

```python
BETA_CONSTRAINT = "nonnegative"
```

to enforce \(\beta_r\ge 0\). This uses the parametrization

$$
\beta_r=\operatorname{softplus}(\eta_r).
$$


In [28]:
BETA_CONSTRAINT = "unconstrained"  # one of {"unconstrained", "nonnegative"}


def inverse_softplus(y):
    y = torch.as_tensor(y, dtype=torch.float32)
    return y + torch.log(-torch.expm1(-y))


class SimilarityWeightedLinearMPLE(nn.Module):
    def __init__(self, n_features_aug, n_relations=3, beta_constraint="unconstrained"):
        super().__init__()

        if beta_constraint not in {"unconstrained", "nonnegative"}:
            raise ValueError("beta_constraint must be 'unconstrained' or 'nonnegative'.")

        self.beta_constraint = beta_constraint

        if beta_constraint == "unconstrained":
            self.beta_param = nn.Parameter(torch.zeros(n_relations))
        else:
            # Initialize beta close to zero but positive.
            beta_init = torch.full((n_relations,), 1e-4)
            self.beta_param = nn.Parameter(inverse_softplus(beta_init))

        self.gamma = nn.Parameter(torch.zeros(n_features_aug))

    @property
    def beta(self):
        if self.beta_constraint == "unconstrained":
            return self.beta_param
        return F.softplus(self.beta_param)

    def logits(self, Sbar_w, X_aug):
        h = Sbar_w @ self.beta + 2.0 * (X_aug @ self.gamma)
        return 2.0 * h

    def forward(self, Sbar_w, X_aug):
        return self.logits(Sbar_w, X_aug)


## Fit the model

The ordinary pseudo-likelihood uses `USE_POS_WEIGHT = False`.

If the main target is minority-class F1 rather than parameter interpretation, try `USE_POS_WEIGHT = True`. This changes the objective, so the result is a weighted pseudo-likelihood classifier rather than the exact MPLE.


In [29]:
def fit_similarity_weighted_linear_mple(
    Sbar_w,
    X_aug,
    y01,
    train_mask,
    beta_constraint="unconstrained",
    l2_beta=1e-4,
    l2_gamma=1e-4,
    max_iter=250,
    pos_weight=None,
    device=None,
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    S_dev = Sbar_w.to(device).float()
    X_dev = X_aug.to(device).float()
    y_dev = y01.to(device).float()
    train_mask_dev = train_mask.to(device).bool()

    model = SimilarityWeightedLinearMPLE(
        n_features_aug=X_aug.shape[1],
        n_relations=Sbar_w.shape[1],
        beta_constraint=beta_constraint,
    ).to(device)

    if pos_weight is not None:
        pos_weight_tensor = torch.tensor(float(pos_weight), device=device)
    else:
        pos_weight_tensor = None

    optimizer = torch.optim.LBFGS(
        model.parameters(),
        lr=1.0,
        max_iter=max_iter,
        line_search_fn="strong_wolfe",
    )

    def objective():
        logits = model(S_dev, X_dev)

        data_loss = F.binary_cross_entropy_with_logits(
            logits[train_mask_dev],
            y_dev[train_mask_dev],
            pos_weight=pos_weight_tensor,
        )

        beta = model.beta
        penalty = 0.5 * l2_beta * (beta @ beta)

        # Do not penalize the intercept gamma[0].
        if model.gamma.numel() > 1:
            penalty = penalty + 0.5 * l2_gamma * (model.gamma[1:] @ model.gamma[1:])

        return data_loss + penalty

    def closure():
        optimizer.zero_grad(set_to_none=True)
        loss = objective()
        loss.backward()
        return loss

    optimizer.step(closure)

    with torch.no_grad():
        final_loss = objective().item()

    return model, final_loss


In [30]:
USE_POS_WEIGHT = False

pos_weight = None
if USE_POS_WEIGHT:
    y_train = y01[train_mask].float()
    pos_weight = (y_train.numel() - y_train.sum()).item() / max(float(y_train.sum().item()), 1.0)
    print("Using pos_weight:", pos_weight)

model, final_loss = fit_similarity_weighted_linear_mple(
    Sbar_w=Sbar_w,
    X_aug=X_aug,
    y01=y01,
    train_mask=train_mask,
    beta_constraint=BETA_CONSTRAINT,
    l2_beta=1e-4,
    l2_gamma=1e-4,
    max_iter=250,
    pos_weight=pos_weight,
    device=device,
)

print("beta constraint:", BETA_CONSTRAINT)
print("Final regularized training objective:", final_loss)


beta constraint: unconstrained
Final regularized training objective: 0.22401392459869385


## Estimated parameters


In [31]:
@torch.no_grad()
def convert_gamma_to_raw_feature_scale(model, X_mean, X_std):
    gamma_std_scale = model.gamma.detach().cpu()

    gamma_raw_scale = torch.empty_like(gamma_std_scale)
    gamma_raw_scale[1:] = gamma_std_scale[1:] / X_std.cpu()
    gamma_raw_scale[0] = gamma_std_scale[0] - (gamma_std_scale[1:] * X_mean.cpu() / X_std.cpu()).sum()

    return gamma_std_scale, gamma_raw_scale


with torch.no_grad():
    beta_hat = model.beta.detach().cpu()
    beta_param_hat = model.beta_param.detach().cpu()

gamma_std_scale, gamma_raw_scale = convert_gamma_to_raw_feature_scale(
    model=model,
    X_mean=X_mean,
    X_std=X_std,
)

beta_df = pd.DataFrame({
    "parameter": BETA_LABELS,
    "estimate_beta": beta_hat.numpy(),
    "internal_beta_param": beta_param_hat.numpy(),
    "constraint": BETA_CONSTRAINT,
})

beta_df


,parameter,estimate_beta,internal_beta_param,constraint
0,beta_1: net_rur,2.167182,2.167182,unconstrained
1,beta_2: net_rsr,1.344409,1.344409,unconstrained
2,beta_3: net_rtr,0.465728,0.465728,unconstrained


In [32]:
gamma_df = pd.DataFrame({
    "parameter": ["gamma_intercept"] + [f"gamma_feature_{j}" for j in range(X_raw.shape[1])],
    "estimate_standardized_X_scale": gamma_std_scale.numpy(),
    "estimate_raw_X_scale": gamma_raw_scale.numpy(),
})

gamma_df.head(15)


,parameter,estimate_standardized_X_scale,estimate_raw_X_scale
0,gamma_intercept,0.175873,-2.801896
1,gamma_feature_0,0.068573,0.235935
2,gamma_feature_1,0.017688,0.061224
3,gamma_feature_2,-0.037273,-0.131973
4,gamma_feature_3,-0.036317,-0.181857
5,gamma_feature_4,-0.028027,-0.191438
6,gamma_feature_5,-0.110080,-0.371502
7,gamma_feature_6,0.011920,0.039203
8,gamma_feature_7,-0.026104,-0.090345
9,gamma_feature_8,-0.030218,-0.104777


## Prediction and evaluation


In [33]:
@torch.no_grad()
def predict_prob(model, Sbar_w, X_aug, device=None):
    if device is None:
        device = next(model.parameters()).device

    model.eval()
    logits = model(Sbar_w.to(device).float(), X_aug.to(device).float())
    return torch.sigmoid(logits).detach().cpu(), logits.detach().cpu()


def safe_auc(y_true, score):
    y_true = np.asarray(y_true)
    if np.unique(y_true).size < 2:
        return np.nan
    return roc_auc_score(y_true, score)


def safe_ap(y_true, score):
    y_true = np.asarray(y_true)
    if np.unique(y_true).size < 2:
        return np.nan
    return average_precision_score(y_true, score)


def evaluate_mask(name, mask, y01, prob, threshold=0.5):
    idx = mask.cpu().numpy().astype(bool)
    y_true = y01.cpu().numpy()[idx].astype(int)
    score = prob.cpu().numpy()[idx]
    y_pred = (score >= threshold).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    return {
        "split": name,
        "threshold": float(threshold),
        "n": int(idx.sum()),
        "positive_rate": float(y_true.mean()),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_pos": precision_score(y_true, y_pred, zero_division=0),
        "recall_pos": recall_score(y_true, y_pred, zero_division=0),
        "f1_pos": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": safe_auc(y_true, score),
        "average_precision": safe_ap(y_true, score),
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1]),
    }


def select_threshold_on_validation(y01, prob, val_mask, objective="balanced_accuracy", n_grid=1001):
    y_true = y01.cpu().numpy()[val_mask.cpu().numpy().astype(bool)].astype(int)
    score = prob.cpu().numpy()[val_mask.cpu().numpy().astype(bool)]

    thresholds = np.linspace(0.0, 1.0, n_grid)
    best_t = 0.5
    best_value = -np.inf

    for t in thresholds:
        pred = (score >= t).astype(int)

        if objective == "balanced_accuracy":
            value = balanced_accuracy_score(y_true, pred)
        elif objective == "f1":
            value = f1_score(y_true, pred, zero_division=0)
        else:
            raise ValueError("objective must be 'balanced_accuracy' or 'f1'.")

        if value > best_value:
            best_value = value
            best_t = float(t)

    return best_t, best_value


In [34]:
prob, logits = predict_prob(model, Sbar_w, X_aug, device=device)

metrics_05 = pd.DataFrame([
    evaluate_mask("train", train_mask, y01, prob, threshold=0.5),
    evaluate_mask("validation", val_mask, y01, prob, threshold=0.5),
    evaluate_mask("test", test_mask, y01, prob, threshold=0.5),
])

metrics_05


,split,threshold,n,positive_rate,accuracy,balanced_accuracy,precision_pos,recall_pos,f1_pos,roc_auc,average_precision,tn,fp,fn,tp
0,train,0.5,32167,0.146921,0.909379,0.751239,0.785377,0.527296,0.630966,0.927411,0.741116,26760,681,2234,2492
1,validation,0.5,4595,0.141676,0.900979,0.726849,0.725806,0.483871,0.580645,0.923222,0.719965,3825,119,336,315
2,test,0.5,9192,0.141427,0.909595,0.741424,0.776207,0.506923,0.613309,0.930245,0.729959,7702,190,641,659


In [35]:
best_t_balacc, best_val_balacc = select_threshold_on_validation(
    y01=y01,
    prob=prob,
    val_mask=val_mask,
    objective="balanced_accuracy",
)

best_t_f1, best_val_f1 = select_threshold_on_validation(
    y01=y01,
    prob=prob,
    val_mask=val_mask,
    objective="f1",
)

print("Best validation threshold for balanced accuracy:", best_t_balacc, "value:", best_val_balacc)
print("Best validation threshold for F1:", best_t_f1, "value:", best_val_f1)


Best validation threshold for balanced accuracy: 0.176 value: 0.8424404800852487
Best validation threshold for F1: 0.269 value: 0.6518415566365532


In [36]:
metrics_val_balacc = pd.DataFrame([
    evaluate_mask("train", train_mask, y01, prob, threshold=best_t_balacc),
    evaluate_mask("validation", val_mask, y01, prob, threshold=best_t_balacc),
    evaluate_mask("test", test_mask, y01, prob, threshold=best_t_balacc),
])

metrics_val_balacc


,split,threshold,n,positive_rate,accuracy,balanced_accuracy,precision_pos,recall_pos,f1_pos,roc_auc,average_precision,tn,fp,fn,tp
0,train,0.176,32167,0.146921,0.857773,0.842374,0.509928,0.820567,0.628984,0.927411,0.741116,23714,3727,848,3878
1,validation,0.176,4595,0.141676,0.858324,0.842440,0.500000,0.820276,0.621291,0.923222,0.719965,3410,534,117,534
2,test,0.176,9192,0.141427,0.862163,0.846803,0.507809,0.825385,0.628772,0.930245,0.729959,6852,1040,227,1073


In [37]:
metrics_val_f1 = pd.DataFrame([
    evaluate_mask("train", train_mask, y01, prob, threshold=best_t_f1),
    evaluate_mask("validation", val_mask, y01, prob, threshold=best_t_f1),
    evaluate_mask("test", test_mask, y01, prob, threshold=best_t_f1),
])

metrics_val_f1


,split,threshold,n,positive_rate,accuracy,balanced_accuracy,precision_pos,recall_pos,f1_pos,roc_auc,average_precision,tn,fp,fn,tp
0,train,0.269,32167,0.146921,0.891784,0.821935,0.611379,0.723022,0.662530,0.927411,0.741116,25269,2172,1309,3417
1,validation,0.269,4595,0.141676,0.890968,0.819774,0.595178,0.720430,0.651842,0.923222,0.719965,3625,319,182,469
2,test,0.269,9192,0.141427,0.889795,0.816312,0.591460,0.713846,0.646915,0.930245,0.729959,7251,641,372,928


## Compare unconstrained and nonnegative beta fits

This cell fits both versions with the same weighted averages and reports their validation/test metrics. It is useful because different Yelp relations may be assortative or disassortative.


In [38]:
def fit_and_evaluate_constraint(beta_constraint):
    model_c, final_loss_c = fit_similarity_weighted_linear_mple(
        Sbar_w=Sbar_w,
        X_aug=X_aug,
        y01=y01,
        train_mask=train_mask,
        beta_constraint=beta_constraint,
        l2_beta=1e-4,
        l2_gamma=1e-4,
        max_iter=250,
        pos_weight=pos_weight,
        device=device,
    )

    prob_c, _ = predict_prob(model_c, Sbar_w, X_aug, device=device)

    t_c, val_balacc_c = select_threshold_on_validation(
        y01=y01,
        prob=prob_c,
        val_mask=val_mask,
        objective="balanced_accuracy",
    )

    metrics_c = pd.DataFrame([
        evaluate_mask("validation", val_mask, y01, prob_c, threshold=t_c),
        evaluate_mask("test", test_mask, y01, prob_c, threshold=t_c),
    ])

    beta_c = model_c.beta.detach().cpu().numpy()

    metrics_c["constraint"] = beta_constraint
    metrics_c["final_loss"] = final_loss_c
    metrics_c["selected_threshold"] = t_c

    for k, label in enumerate(["beta_1", "beta_2", "beta_3"]):
        metrics_c[label] = beta_c[k]

    return model_c, metrics_c


model_unconstrained, metrics_unconstrained = fit_and_evaluate_constraint("unconstrained")
model_nonnegative, metrics_nonnegative = fit_and_evaluate_constraint("nonnegative")

comparison_df = pd.concat([metrics_unconstrained, metrics_nonnegative], ignore_index=True)
comparison_df


,split,threshold,n,positive_rate,accuracy,balanced_accuracy,precision_pos,recall_pos,f1_pos,roc_auc,...,tn,fp,fn,tp,constraint,final_loss,selected_threshold,beta_1,beta_2,beta_3
0,validation,0.176,4595,0.141676,0.858324,0.842440,0.500000,0.820276,0.621291,0.923222,...,3410,534,117,534,unconstrained,0.224014,0.176,2.167182,1.344409,0.465728
1,test,0.176,9192,0.141427,0.862163,0.846803,0.507809,0.825385,0.628772,0.930245,...,6852,1040,227,1073,unconstrained,0.224014,0.176,2.167182,1.344409,0.465728
2,validation,0.114,4595,0.141676,0.639391,0.712340,0.256534,0.814132,0.390136,0.772184,...,2408,1536,121,530,nonnegative,0.350953,0.114,0.000102,0.000101,0.000101
3,test,0.114,9192,0.141427,0.642950,0.698902,0.252374,0.776923,0.380988,0.775312,...,4900,2992,290,1010,nonnegative,0.350953,0.114,0.000102,0.000101,0.000101


## Local field decomposition

For each node,

$$
h_i
=
\underbrace{\sum_{r=1}^3\beta_r\overline S^{\,w}_{ir}}_{\text{weighted graph field}}
+
\underbrace{2X_i^\top\gamma}_{\text{external feature field}}.
$$


In [39]:
@torch.no_grad()
def local_field_decomposition(model, Sbar_w, X_aug, y01, train_mask, val_mask, test_mask, device):
    model.eval()

    S_dev = Sbar_w.to(device).float()
    X_dev = X_aug.to(device).float()

    beta = model.beta.detach()
    gamma = model.gamma.detach()

    graph_field = S_dev @ beta
    feature_field = 2.0 * (X_dev @ gamma)
    total_h = graph_field + feature_field
    prob = torch.sigmoid(2.0 * total_h)

    return pd.DataFrame({
        "graph_field": graph_field.cpu().numpy(),
        "feature_field_2Xgamma": feature_field.cpu().numpy(),
        "total_h": total_h.cpu().numpy(),
        "prob_label_1": prob.cpu().numpy(),
        "label": y01.numpy(),
        "train": train_mask.numpy(),
        "validation": val_mask.numpy(),
        "test": test_mask.numpy(),
    })


field_decomp_df = local_field_decomposition(
    model=model,
    Sbar_w=Sbar_w,
    X_aug=X_aug,
    y01=y01,
    train_mask=train_mask,
    val_mask=val_mask,
    test_mask=test_mask,
    device=device,
)

field_decomp_df.head()


,graph_field,feature_field_2Xgamma,total_h,prob_label_1,label,train,validation,test
0,-1.344409,0.488444,-0.855965,0.152914,0,False,True,False
1,0.000000,0.785456,0.785456,0.827914,0,True,False,False
2,0.000000,1.093967,1.093967,0.899161,0,True,False,False
3,1.344409,0.472672,1.817081,0.974273,0,False,False,True
4,-1.344409,0.363980,-0.980428,0.123374,0,False,False,True


In [40]:
field_decomp_df.groupby("label")[[
    "graph_field",
    "feature_field_2Xgamma",
    "total_h",
    "prob_label_1",
]].describe()


graph_field                                                              \
            count      mean       std       min       25%       50%       75%   
label                                                                           
0         39277.0 -2.370988  1.143858 -3.977318 -3.592948 -1.764217 -1.444083   
1          6677.0 -0.504560  0.954279 -3.977318 -1.180046 -0.751648  0.000000   

                feature_field_2Xgamma           ...   total_h            \
            max                 count     mean  ...       75%       max   
label                                           ...                       
0      2.327473               39277.0  0.28798  ... -1.138496  2.604186   
1      3.977318                6677.0  0.72396  ...  0.930968  5.118335   

      prob_label_1                                                    \
             count      mean       std       min       25%       50%   
label                                                                  
0          39277.0  0.079234  0.136798  0.000083  0.001582  0.024704   
1           6677.0  0.537377  0.321010  0.000228  0.238681  0.519870   

                           
            75%       max  
label                      
0      0.093046  0.994559  
1      0.865523  0.999964  

[2 rows x 32 columns]

## Optional: save outputs


In [ ]:
# Edit ROW_NAME before running if you want a custom label in the CSV.
from pathlib import Path

ROW_NAME = "S_hm_weighted_linear"

# For Colab, set SAVE_TO_GOOGLE_DRIVE=True and edit GOOGLE_DRIVE_METRICS_CSV_PATH.
SAVE_TO_GOOGLE_DRIVE = False
GOOGLE_DRIVE_METRICS_CSV_PATH = "/content/drive/MyDrive/yelp_s_metrics.csv"

if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    METRICS_CSV_PATH = Path(GOOGLE_DRIVE_METRICS_CSV_PATH)
else:
    METRICS_CSV_PATH = Path("script/yelp/yelp_s_metrics.csv") if Path("script/yelp").exists() else Path("yelp_s_metrics.csv")


def append_test_metric_row(row_name, metric_frames, extra_metadata, csv_path=METRICS_CSV_PATH):
    csv_path = Path(csv_path)
    row = {"row_name": row_name, **extra_metadata}

    for metric_set, frame in metric_frames.items():
        test_rows = frame.loc[frame["split"].astype(str).str.lower().eq("test")]
        if test_rows.empty:
            raise ValueError(f"No test split found in {metric_set}")

        test_row = test_rows.iloc[0]
        prefix = metric_set.replace(".", "_")
        for column, value in test_row.items():
            if column == "split":
                continue
            row[f"{prefix}_{column}"] = value

    saved = pd.DataFrame([row])
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    saved.to_csv(csv_path, mode="a", header=not csv_path.exists(), index=False)
    print(f"Saved 1 test metric row to {csv_path}")
    return saved


metric_frames = {
    "threshold_0.5": metrics_05,
    "threshold_val_balanced_accuracy": metrics_val_balacc,
    "threshold_val_f1": metrics_val_f1,
}

run_metadata = {
    "notebook": "S_hm_weighted_linear.ipynb",
    "model_type": "similarity_weighted_linear_mple",
    "similarity_mode": SIMILARITY_MODE,
    "beta_constraint": BETA_CONSTRAINT,
    "denominator_observed_only": DENOMINATOR_USES_ONLY_OBSERVED_LABELS,
    "source_labels": "train_mask_only" if torch.equal(SOURCE_LABEL_MASK_FOR_NEIGHBOR_AVERAGES, train_mask) else "custom",
    "selected_threshold_balanced_accuracy": best_t_balacc,
    "selected_threshold_f1": best_t_f1,
}

saved_metric_row = append_test_metric_row(ROW_NAME, metric_frames, run_metadata)
saved_metric_row


In [41]:
results = {
    "model_type": "similarity_weighted_linear_mple",
    "similarity_mode": SIMILARITY_MODE,
    "edge_types": EDGE_TYPES,
    "beta_constraint": BETA_CONSTRAINT,
    "denominator_observed_only": DENOMINATOR_USES_ONLY_OBSERVED_LABELS,
    "source_labels_for_neighbor_averages": (
        "train_mask_only"
        if torch.equal(SOURCE_LABEL_MASK_FOR_NEIGHBOR_AVERAGES, train_mask)
        else "custom"
    ),
    "tau_values": tau_values.numpy().tolist(),
    "beta_hat": beta_hat.numpy().tolist(),
    "gamma_standardized_X_scale": gamma_std_scale.numpy().tolist(),
    "gamma_raw_X_scale": gamma_raw_scale.numpy().tolist(),
    "threshold_05_metrics": metrics_05.to_dict(orient="records"),
    "threshold_val_balanced_accuracy": best_t_balacc,
    "threshold_val_balanced_accuracy_metrics": metrics_val_balacc.to_dict(orient="records"),
    "threshold_val_f1": best_t_f1,
    "threshold_val_f1_metrics": metrics_val_f1.to_dict(orient="records"),
}

pd.Series(results)


,0
model_type,similarity_weighted_linear_mple
similarity_mode,cosine_positive
edge_types,"[net_rur, net_rsr, net_rtr]"
beta_constraint,unconstrained
denominator_observed_only,True
source_labels_for_neighbor_averages,train_mask_only
tau_values,"[6.746762752532959, 7.5904412269592285, 7.5698..."
beta_hat,"[2.167181968688965, 1.3444087505340576, 0.4657..."
gamma_standardized_X_scale,"[0.175873264670372, 0.06857255846261978, 0.017..."
gamma_raw_X_scale,"[-2.801896095275879, 0.23593482375144958, 0.06..."
